In [1]:
import csv
from pathlib import Path

In [2]:
actual_path = Path("raw-zip-actual")
actual_files = sorted(actual_path.glob("*.zip"))
print(len(actual_files))
actual_files[:5]

230


[PosixPath('raw-zip-actual/20060101RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060201RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060301RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060401RTLineOutages_csv.zip'),
 PosixPath('raw-zip-actual/20060501RTLineOutages_csv.zip')]

In [3]:
scheduled_path = Path("raw-zip-scheduled")
scheduled_files = sorted(scheduled_path.glob("*.zip"))
print(len(scheduled_files))
scheduled_files[:5]

254


[PosixPath('raw-zip-scheduled/20050201SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050301SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050401SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050501SCLineOutages_csv.zip'),
 PosixPath('raw-zip-scheduled/20050601SCLineOutages_csv.zip')]

In [4]:
zip_path = actual_files[0]

In [5]:
import io
from datetime import datetime
from zipfile import ZipFile


def dont_parse_data_LOL(row):
    return row


def list_csvs(zip_path):
    with ZipFile(zip_path) as z:
        return [i for i in sorted(z.namelist()) if i.endswith(".csv")]


def read_csv_from_zip(zip_path, csv_name, parse_row):
    with ZipFile(zip_path) as z:
        with z.open(csv_name) as f:
            data = f.read()
        csv_reader = csv.DictReader(io.StringIO(data.decode("utf-8")))
        # Only keep rows with line outages
        data = [parse_row(row) for row in csv_reader]
        data = [row for row in data if row]
    return data


# Example using an existing variable in the notebook:
zip_path = actual_files[100]
print(zip_path)
print("members:", list_csvs(zip_path))

# Read one file from the zip (text)
member = list_csvs(zip_path)[0]
data = read_csv_from_zip(zip_path, member, dont_parse_data_LOL)
print(len(data))

raw-zip-actual/20150501RTLineOutages_csv.zip
members: ['20150501RTLineOutages.csv', '20150502RTLineOutages.csv', '20150503RTLineOutages.csv', '20150504RTLineOutages.csv', '20150505RTLineOutages.csv', '20150506RTLineOutages.csv', '20150507RTLineOutages.csv', '20150508RTLineOutages.csv', '20150509RTLineOutages.csv', '20150510RTLineOutages.csv', '20150511RTLineOutages.csv', '20150512RTLineOutages.csv', '20150513RTLineOutages.csv', '20150514RTLineOutages.csv', '20150515RTLineOutages.csv', '20150516RTLineOutages.csv', '20150517RTLineOutages.csv', '20150518RTLineOutages.csv', '20150519RTLineOutages.csv', '20150520RTLineOutages.csv', '20150521RTLineOutages.csv', '20150522RTLineOutages.csv', '20150523RTLineOutages.csv', '20150524RTLineOutages.csv', '20150525RTLineOutages.csv', '20150526RTLineOutages.csv', '20150527RTLineOutages.csv', '20150528RTLineOutages.csv', '20150529RTLineOutages.csv', '20150530RTLineOutages.csv', '20150531RTLineOutages.csv']
30991


In [6]:
from tqdm import tqdm
import re

equipment_name_pattern = r"^([A-Za-z0-9._ -]{8})-([A-Za-z0-9._ -]{8})_(\d{2,3})_(.+)$"
actual_outage_len = []
unique_device = set()
unique_lines = set()
line_counter = 0
for zip_path in tqdm(actual_files):
    for member in list_csvs(zip_path):
        data = read_csv_from_zip(zip_path, member, dont_parse_data_LOL)
        actual_outage_len.append(len(data))
        for row in data:
            unique_device.add(row["PTID"])
            m = re.match(equipment_name_pattern, row["Equipment Name"])
            if m is not None:
                line_counter += 1
                unique_lines.add(m.group(1))
                unique_lines.add(m.group(2))

print(len(actual_outage_len))
print(f"Unique devices: {len(unique_device)}")

100%|██████████| 230/230 [07:17<00:00,  1.90s/it]

6995
Unique devices: 5161


In [15]:
print(f"{line_counter:,}")
print(f"Unique lines: {len(unique_lines)}")

115,362,978
Unique lines: 1456


In [7]:
data_len = len(actual_outage_len)
lower = data_len // 4
upper = 3 * data_len // 4

In [8]:
# average length of the middle 50% of the logs per day
sum(sorted(actual_outage_len)[lower:upper]) / (upper - lower)

39221.10548885077

In [9]:
sum(actual_outage_len[lower:upper]) / (upper - lower)

35389.12721555174

In [10]:
actual_files[22]

PosixPath('raw-zip-actual/20081101RTLineOutages_csv.zip')

In [11]:
# average length of the logs per day after November 2008
sum(actual_outage_len[22:]) / (data_len - 22)

39369.17323963861

In [12]:
# compare number of logs for: all element vs lines
number_of_outages_for_all_logs = sum(actual_outage_len)
print(f"Number of outages for all logs: {number_of_outages_for_all_logs:,}")

Number of outages for all logs: 275,643,011
